# Financial_LLM_FineTuning
## Financial Domain Fine-Tuning with LoRA

This notebook fine-tunes `HuggingFaceTB/SmolLM2-360M-Instruct` on a filtered subset of `sujet-ai/Sujet-Finance-Instruct-177k`.

### Objective

Build a lightweight financial-domain conversational model using:

- Supervised fine-tuning
- LoRA / PEFT
- Hugging Face Transformers
- Hugging Face Datasets

### Pipeline

Dataset → Filtering → Train/Test Split → Train-only Augmentation → Chat Formatting → Tokenization → LoRA → Fine-Tuning → Evaluation → Merge → Inference → Hugging Face Hub

**Hardware target:** Google Colab GPU, including NVIDIA T4.

## 1. Environment Setup

This project requires a matching PyTorch/TorchVision CUDA build. The previous error came from PyTorch being built for CUDA 13.0 while TorchVision was built for CUDA 12.8.

Use a fresh Colab runtime for this notebook.

**After this cell finishes, restart the runtime once, then continue from Cell 4.**

In [1]:
# Install a matching PyTorch/TorchVision pair.
# PyTorch 2.11.0 + TorchVision 0.26.0 + CUDA 13.0
# are an officially supported combination.

!pip install -q --no-cache-dir \
    torch==2.11.0 \
    torchvision==0.26.0 \
    --index-url https://download.pytorch.org/whl/cu130

# Project dependencies
!pip install -q \
    transformers==4.47.1 \
    datasets==3.2.0 \
    peft==0.14.0 \
    huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Imports and Environment Verification

In [1]:
import math
import re
import os

import torch
import torchvision

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
else:
    print("WARNING: CUDA is not available.")

PyTorch: 2.11.0+cu128
TorchVision: 0.26.0+cu128
CUDA: 12.8
CUDA available: True
GPU: Tesla T4
GPU capability: (7, 5)


## 3. Configuration

In [2]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
DATASET_NAME = "sujet-ai/Sujet-Finance-Instruct-177k"

OUTPUT_DIR = "./lora_finetuned"
MERGED_MODEL_DIR = "./FinChat-XS"

MAX_LENGTH = 512
TEST_SIZE = 0.10
SEED = 42

NUM_EPOCHS = 1
LEARNING_RATE = 1.5e-4

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

DUPLICATION_FACTOR = 5

print("Model:", MODEL_NAME)
print("Dataset:", DATASET_NAME)
print("Max sequence length:", MAX_LENGTH)
print("Seed:", SEED)

Model: HuggingFaceTB/SmolLM2-360M-Instruct
Dataset: sujet-ai/Sujet-Finance-Instruct-177k
Max sequence length: 512
Seed: 42


## 4. Load the Financial Dataset

In [3]:
dataset = load_dataset(DATASET_NAME)

print(dataset)
print("Original training examples:", len(dataset["train"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sujet-ai/Sujet-Finance-Instruct-177k.csv:   0%|          | 0.00/337M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/177597 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'inputs', 'answer', 'system_prompt', 'user_prompt', 'task_type', 'dataset', 'index_level', 'conversation_id'],
        num_rows: 177597
    })
})
Original training examples: 177597


In [4]:
# Keep only the QA and conversational QA categories used by the
# original FinChat-XS training pipeline.

allowed_task_types = {
    "qa",
    "qa_conversation",
}

dataset["train"] = dataset["train"].filter(
    lambda x: x["task_type"].strip().lower() in allowed_task_types
)

print("After task-type filtering:", len(dataset["train"]))

Filter:   0%|          | 0/177597 [00:00<?, ? examples/s]

After task-type filtering: 54414


In [5]:
# Restrict prompt and answer length for the 360M parameter model.

def filter_by_length(example):
    user_prompt = str(example["user_prompt"])
    answer = str(example["answer"])

    return (
        len(answer) <= 300
        and len(user_prompt) <= 100
    )

dataset["train"] = dataset["train"].filter(filter_by_length)

print("After length filtering:", len(dataset["train"]))

Filter:   0%|          | 0/54414 [00:00<?, ? examples/s]

After length filtering: 15752


In [6]:
# Remove examples containing excessive emojis or highly informal punctuation.

emoji_pattern = re.compile(
    "["
    u"\U0001F600-\U0001F64F"
    u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F6FF"
    u"\U0001F700-\U0001F77F"
    u"\U0001F780-\U0001F7FF"
    u"\U0001F800-\U0001F8FF"
    u"\U0001F900-\U0001F9FF"
    u"\U0001FA00-\U0001FA6F"
    u"\U0001FA70-\U0001FAFF"
    u"\U00002702-\U000027B0"
    "]+",
    flags=re.UNICODE
)

def clean_dataset(example):
    answer = str(example["answer"])

    if emoji_pattern.search(answer):
        return False

    if re.search(r"[!?]{3,}", answer):
        return False

    return True

dataset["train"] = dataset["train"].filter(clean_dataset)

print("After content cleaning:", len(dataset["train"]))

Filter:   0%|          | 0/15752 [00:00<?, ? examples/s]

After content cleaning: 15749


## 5. Train/Test Split

The split is performed **before** augmentation.

This prevents duplicated conversational examples from leaking into the evaluation set.

In [7]:
split_dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    seed=SEED
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(test_dataset))

Training examples: 14174
Evaluation examples: 1575


## 6. Training-Only Conversation Augmentation

In [8]:
short_qa_conversations = []

for i, example in enumerate(train_dataset):
    task_type = example["task_type"].strip().lower()
    answer = str(example["answer"])

    if task_type == "qa_conversation" and len(answer) < 30:
        short_qa_conversations.append(i)

print(
    f"Found {len(short_qa_conversations)} short "
    "qa_conversation examples."
)

Found 5 short qa_conversation examples.


In [9]:
# Duplicate short conversational examples only in the training split.

augmented_train_dataset = train_dataset

for _ in range(DUPLICATION_FACTOR - 1):
    for idx in short_qa_conversations:
        augmented_train_dataset = augmented_train_dataset.add_item(
            train_dataset[idx]
        )

split_dataset = {
    "train": augmented_train_dataset,
    "test": test_dataset,
}

print("Original training size:", len(train_dataset))
print("Augmented training size:", len(augmented_train_dataset))
print(
    "Added:",
    len(augmented_train_dataset) - len(train_dataset),
    "training examples"
)

Original training size: 14174
Augmented training size: 14194
Added: 20 training examples


In [10]:
def count_task_types(ds):
    qa = 0
    qa_conversation = 0

    for example in ds:
        task_type = example["task_type"].strip().lower()

        if task_type == "qa":
            qa += 1
        elif task_type == "qa_conversation":
            qa_conversation += 1

    return qa, qa_conversation

train_qa, train_conv = count_task_types(split_dataset["train"])
test_qa, test_conv = count_task_types(split_dataset["test"])

print(f"Train: {train_qa} QA | {train_conv} QA conversation")
print(f"Test:  {test_qa} QA | {test_conv} QA conversation")

Train: 14142 QA | 52 QA conversation
Test:  1573 QA | 2 QA conversation


## 7. Load Tokenizer and Base Model

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Tokenizer loaded.
Vocabulary size: 49152
PAD token: <|im_end|>
EOS token: <|im_end|>


In [12]:
# Select a precision supported by the available GPU.
#
# T4 -> FP16
# A100 / newer BF16-capable GPUs -> BF16
# CPU -> FP32

if torch.cuda.is_available():
    gpu_major = torch.cuda.get_device_capability(0)[0]

    if gpu_major >= 8:
        MODEL_DTYPE = torch.bfloat16
    else:
        MODEL_DTYPE = torch.float16
else:
    MODEL_DTYPE = torch.float32

print("Selected model dtype:", MODEL_DTYPE)

Selected model dtype: torch.float16


In [13]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Device:", device)
print("Model dtype:", model.dtype)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Device: cuda
Model dtype: torch.float16
GPU: Tesla T4


## 8. Format the Dataset Using the Model Chat Template

In [14]:
def format_chat(example):
    user_prompt = str(example["user_prompt"])

    # Remove the "Question:" prefix when present.
    if re.match(
        r"^\s*question\b",
        user_prompt,
        re.IGNORECASE
    ):
        user_prompt = re.sub(
            r"^\s*question.*?:\s*",
            "",
            user_prompt,
            flags=re.IGNORECASE
        )

    system_prompt = str(example.get("system_prompt", "")).strip()

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.extend([
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": str(example["answer"])
        }
    ])

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

In [15]:
formatted_train = split_dataset["train"].map(
    format_chat
)

formatted_test = split_dataset["test"].map(
    format_chat
)

formatted_dataset = {
    "train": formatted_train,
    "test": formatted_test,
}

print(formatted_dataset)

Map:   0%|          | 0/14194 [00:00<?, ? examples/s]

Map:   0%|          | 0/1575 [00:00<?, ? examples/s]

{'train': Dataset({
    features: ['Unnamed: 0', 'inputs', 'answer', 'system_prompt', 'user_prompt', 'task_type', 'dataset', 'index_level', 'conversation_id', 'text'],
    num_rows: 14194
}), 'test': Dataset({
    features: ['Unnamed: 0', 'inputs', 'answer', 'system_prompt', 'user_prompt', 'task_type', 'dataset', 'index_level', 'conversation_id', 'text'],
    num_rows: 1575
})}


In [16]:
# Inspect one formatted example.

sample_index = min(
    2000,
    len(formatted_dataset["train"]) - 1
)

print(formatted_dataset["train"][sample_index]["text"])

<|im_start|>system
As a finance expert, your role is to provide clear, concise, and informative responses to finance-related questions. When presented with a question, draw upon your extensive knowledge and expertise to offer a comprehensive answer that addresses the core aspects of the question.<|im_end|>
<|im_start|>user
Generate a list of the top five actors from the movie Titanic.<|im_end|>
<|im_start|>assistant
The top five actors from Titanic are Leonardo DiCaprio, Kate Winslet, Billy Zane, Kathy Bates and Frances Fisher.<|im_end|>



## 9. Tokenization

A fixed 512-token context window is used instead of setting the context length to the longest example in the dataset.

In [17]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

In [18]:
train_columns = formatted_dataset["train"].column_names
test_columns = formatted_dataset["test"].column_names

tokenized_train = formatted_dataset["train"].map(
    tokenize_function,
    batched=True,
    remove_columns=train_columns,
)

tokenized_test = formatted_dataset["test"].map(
    tokenize_function,
    batched=True,
    remove_columns=test_columns,
)

tokenized_dataset = {
    "train": tokenized_train,
    "test": tokenized_test,
}

print(tokenized_dataset)

Map:   0%|          | 0/14194 [00:00<?, ? examples/s]

Map:   0%|          | 0/1575 [00:00<?, ? examples/s]

{'train': Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 14194
}), 'test': Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1575
})}


In [19]:
sample = tokenized_dataset["train"][0]

print("Number of tokens:", len(sample["input_ids"]))
print("First 20 token IDs:", sample["input_ids"][:20])

Number of tokens: 111
First 20 token IDs: [1, 9690, 198, 1653, 253, 9999, 4507, 28, 469, 1791, 314, 288, 1538, 2437, 28, 19484, 28, 284, 17628, 5928]


## 10. LoRA Configuration

LoRA adapts a small fraction of the model parameters while keeping the base model mostly frozen.

In [20]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

model_peft = get_peft_model(
    model,
    lora_config
)

model_peft.print_trainable_parameters()

trainable params: 1,638,400 || all params: 363,459,520 || trainable%: 0.4508


## 11. Training Configuration

In [21]:
# Select mixed precision automatically.

if torch.cuda.is_available():
    gpu_major = torch.cuda.get_device_capability(0)[0]

    USE_BF16 = gpu_major >= 8
    USE_FP16 = gpu_major < 8
else:
    USE_BF16 = False
    USE_FP16 = False

print("FP16:", USE_FP16)
print("BF16:", USE_BF16)

FP16: True
BF16: False


In [22]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    # T4-friendly micro-batch size.
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Effective batch size = 4 x 4 = 16.
    gradient_accumulation_steps=4,

    learning_rate=LEARNING_RATE,
    weight_decay=0.005,

    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.5,

    # T4 -> FP16; newer GPUs -> BF16.
    fp16=USE_FP16,
    bf16=USE_BF16,

    # More broadly compatible than the fused optimizer.
    optim="adamw_torch",

    logging_steps=50,

    # Use the current Transformers argument name.
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    seed=SEED,

    report_to="none",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

## 12. Data Collator and Trainer

In [23]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

print("Data collator ready.")

Data collator ready.


In [24]:
trainer = Trainer(
    model=model_peft,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],

    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3
        )
    ],
)

print("Trainer initialized.")

Trainer initialized.


## 13. Fine-Tune FinChat-XS

In [25]:
train_result = trainer.train()

print("Training complete.")

Step,Training Loss,Validation Loss
100,0.776300,0.738037
200,0.738000,0.718340
300,0.720600,0.711329
400,0.709900,0.706712
500,0.704000,0.703999
600,0.702300,0.702224
700,0.694600,0.701151
800,0.721100,0.700893


Training complete.


In [26]:
# Evaluate the best checkpoint.

evaluation_results = trainer.evaluate()

eval_loss = evaluation_results["eval_loss"]

print(f"Evaluation Loss: {eval_loss:.4f}")

if eval_loss < 20:
    perplexity = math.exp(eval_loss)
    print(f"Perplexity: {perplexity:.4f}")
else:
    perplexity = None
    print("Perplexity is too large to calculate safely.")

Evaluation Loss: 0.7009
Perplexity: 2.0156


## 14. Merge LoRA Adapter and Save the Model

In [27]:
merged_model = model_peft.merge_and_unload()

merged_model.save_pretrained(
    MERGED_MODEL_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    MERGED_MODEL_DIR
)

print(
    "Merged model saved to:",
    MERGED_MODEL_DIR
)

Merged model saved to: ./FinChat-XS


## 15. Financial Inference Test

In [28]:
def generate_response(
    prompt,
    max_new_tokens=150
):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    inputs = inputs.to(device)

    with torch.no_grad():
        outputs = merged_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs.shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

In [29]:
test_prompts = [
    "What is the difference between a stock and a bond?",
    "What is diversification in investing?",
    "Explain market capitalization.",
    "What is the difference between revenue and profit?",
    "What is a dividend?",
]

for prompt in test_prompts:
    print("=" * 80)
    print("USER:", prompt)
    print("MODEL:", generate_response(prompt))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


USER: What is the difference between a stock and a bond?
MODEL: A stock is a share of ownership in a company, while a bond is a loan made to a company or government. The difference is that a stock is a claim on the company's assets and earnings, while a bond is a loan that is repaid with interest. 

In other words, a stock is a way to invest in a company, while a bond is a way to borrow money from a company. The interest on a bond is usually paid out at regular intervals, while the dividend on a stock is paid out at regular intervals. 

For example, if you buy a stock, you own a piece of the company, while if you buy a bond, you are essentially lending money to the company. The interest on a bond
USER: What is diversification in investing?
MODEL: Diversification in investing refers to the practice of spreading investments across different asset classes, sectors, and geographic regions to reduce risk and increase potential returns. This approach helps to reduce the impact of market fluc

## 16. Base vs Fine-Tuned Comparison

This section provides a simple qualitative comparison between the original base model and FinChat-XS.

In [30]:
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
).to(device)

def generate_base_response(
    prompt,
    max_new_tokens=150
):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = base_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = base_model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs.shape[-1]:]

    return base_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

In [31]:
comparison_prompt = (
    "Explain the concept of price-to-earnings ratio "
    "and why investors use it."
)

print("PROMPT:")
print(comparison_prompt)

print("\nBASE MODEL:")
print(generate_base_response(comparison_prompt))

print("\nFINCHAT-XS:")
print(generate_response(comparison_prompt))

PROMPT:
Explain the concept of price-to-earnings ratio and why investors use it.

BASE MODEL:
The price-to-earnings (P/E) ratio is a financial metric used to measure the valuation of a company's stock price relative to its earnings per share. It's calculated by dividing the current stock price by the earnings per share. The P/E ratio is often used by investors to determine whether a stock is undervalued or overvalued.

Investors use the P/E ratio to evaluate a company's stock price in relation to its earnings. A lower P/E ratio indicates that the stock price is lower than the company's earnings, suggesting that investors are willing to pay a premium for the company's earnings. On the other hand, a higher P/E ratio indicates that the stock price is higher than the company

FINCHAT-XS:
The price-to-earnings ratio (P/E ratio) is a financial metric that measures the price of a company's stock relative to its earnings per share. It is calculated by dividing the stock's price by the earnings

# Results

Record the final metrics from the training run here.

### Configuration

| Component | Value |
|---|---|
| Base Model | SmolLM2-360M-Instruct |
| Dataset | Sujet-Finance-Instruct-177k |
| Fine-Tuning | LoRA |
| LoRA Rank | 8 |
| LoRA Alpha | 16 |
| Target Modules | Q/K/V/O |
| Maximum Sequence Length | 512 |
| Epochs | 1 |
| Learning Rate | 1.5e-4 |
| Effective Batch Size | 16 |
| T4 Precision | FP16 |
| Evaluation Split | 10% |

### Metrics

The notebook calculates:

- Evaluation Loss
- Perplexity
- Qualitative inference results
- Base-model vs fine-tuned comparison

The evaluation metrics should be copied from the output after training.

# Conclusion

FinChat-XS is a lightweight financial-domain language model created by parameter-efficient fine-tuning of SmolLM2-360M-Instruct with LoRA.

The pipeline includes data filtering, leakage-safe splitting, training-only augmentation, chat-template formatting, fixed-length tokenization, LoRA fine-tuning, quantitative evaluation, model merging, and inference testing.

The final merged model and LoRA adapter can be uploaded to the Hugging Face Hub for further experimentation and deployment.